# 02 - Delta Lakehouse: Silver MERGE upsert and Gold NEWS2 aggregate
Rubric deliverable 2. Assumes notebook 01 has populated Bronze.

In [1]:
import os
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
os.environ.setdefault('TQDM_DISABLE', '1')

'1'

## 1. Build Silver - real `DeltaTable.merge` keyed on the business key `reading_id`

In [2]:
import shutil
shutil.rmtree('lakehouse/silver', ignore_errors=True)
shutil.rmtree('lakehouse/gold', ignore_errors=True)
from src.lakehouse.silver import build_silver, read_silver
m = build_silver()
print({k: m[k] for k in ('source_rows','num_target_rows_inserted','num_target_rows_updated') if k in m})
print('Silver rows:', read_silver().num_rows)

{'source_rows': 517, 'num_target_rows_inserted': 517, 'num_target_rows_updated': 0}
Silver rows: 517


## 2. A late correction re-sends the same `reading_id` - Silver upserts in place

In [3]:
from datetime import datetime, timezone
from src.lakehouse.bronze import read_bronze, append_bronze

before = read_silver().to_pandas().set_index('reading_id')
ids = list(before.index[:5])
print('BEFORE'); print(before.loc[ids, ['heart_rate','spo2']])

b = read_bronze().to_pandas().set_index('reading_id')
corr = []
for rid in ids:
    r = b.loc[rid].to_dict(); r['reading_id'] = rid
    r['heart_rate'] = int(r['heart_rate']) + 40; r['spo2'] = 93
    r['recorded_at'] = r['recorded_at'].to_pydatetime()
    r['ingested_at'] = datetime.now(timezone.utc)
    corr.append(r)
append_bronze(corr)
m = build_silver()
after = read_silver().to_pandas().set_index('reading_id')
print('\nmerge metrics:', {k: m[k] for k in ('num_target_rows_updated','num_target_rows_inserted') if k in m})
print('Silver rows before/after:', len(before), '/', len(after), '(unchanged = upsert not append)')
print('AFTER'); print(after.loc[ids, ['heart_rate','spo2']])

BEFORE
                                      heart_rate  spo2
reading_id                                            
350fdc5b-8e4f-4a2e-aabf-6db1210ad30a          65    95
85911a59-c9fb-4b67-9808-ccf8d0b25eac          72    98
a4868279-b3c7-4d42-b6f5-1cf38c3a6d05          83    99
bfdccf77-18eb-44bf-80b7-f6b899471ca2          80    97
9503a59b-af4c-4bc8-896c-540eede57511          63    97



merge metrics: {'num_target_rows_updated': 5, 'num_target_rows_inserted': 0}
Silver rows before/after: 517 / 517 (unchanged = upsert not append)
AFTER
                                      heart_rate  spo2
reading_id                                            
350fdc5b-8e4f-4a2e-aabf-6db1210ad30a         105    93
85911a59-c9fb-4b67-9808-ccf8d0b25eac         112    93
a4868279-b3c7-4d42-b6f5-1cf38c3a6d05         123    93
bfdccf77-18eb-44bf-80b7-f6b899471ca2         120    93
9503a59b-af4c-4bc8-896c-540eede57511         103    93


## 3. Build Gold - NEWS2 early-warning aggregate per patient per hour
A genuine reduction: one row per (patient, hour), not a copy of Silver.

In [4]:
from src.lakehouse.gold import build_gold, read_gold
m = build_gold()
print(m)
g = read_gold().to_pandas()
print('\nGold rows:', len(g), ' Silver rows:', m['silver_rows'],
      ' reduction: x%.1f' % (m['silver_rows']/len(g)))
print(g['worst_risk_band'].value_counts().to_dict())
g.sort_values('max_news2', ascending=False).head(8)[
    ['patient_id','window_start','readings_in_window','mean_news2','max_news2','worst_risk_band','min_spo2']]

{'num_target_rows_inserted': 128, 'num_target_rows_updated': 0, 'gold_rows': 128, 'silver_rows': 517}

Gold rows: 128  Silver rows: 517  reduction: x4.0
{'low': 124, 'high': 4}


,patient_id,window_start,readings_in_window,mean_news2,max_news2,worst_risk_band,min_spo2
49,P100007,2026-09-09 15:00:00+00:00,2,12.500,13,high,89
48,P100007,2026-09-09 14:00:00+00:00,5,11.000,13,high,89
47,P100007,2026-09-09 13:00:00+00:00,4,8.000,10,high,92
46,P100007,2026-09-09 12:00:00+00:00,6,4.500,7,high,93
106,P100016,2026-09-09 14:00:00+00:00,5,2.200,4,low,94
25,P100004,2026-09-09 10:00:00+00:00,2,2.000,4,low,93
87,P100013,2026-09-09 14:00:00+00:00,5,1.800,4,low,95
70,P100011,2026-09-09 10:00:00+00:00,3,2.333,4,low,93


### Schema enforcement - delta-rs refuses a wrong-typed write to Bronze

In [5]:
import pyarrow as pa
from deltalake import write_deltalake
from src.lakehouse.bronze import BRONZE_SCHEMA, _table_uri
bad_schema = pa.schema([f if f.name != 'heart_rate'
                        else pa.field('heart_rate', pa.string()) for f in BRONZE_SCHEMA])
row = {f.name: ('x' if f.name in ('reading_id','patient_id','consciousness','source_device')
                else 'fast' if f.name == 'heart_rate' else 1) for f in bad_schema}
import datetime as _dt
for k in ('recorded_at','ingested_at'): row[k] = _dt.datetime.now(_dt.timezone.utc)
row['temperature_c'] = 36.8; row['on_supplemental_o2'] = False
try:
    write_deltalake(_table_uri(), pa.Table.from_pylist([row], schema=bad_schema), mode='append')
    print('ERROR: write was accepted')
except Exception as e:
    print('refused ->', type(e).__name__, str(e)[:160])

refused -> Exception Cast error: Failed to cast heart_rate from Int32 to Utf8: Cannot cast string 'fast' to value of Int32 type
